In [65]:
import requests
from bs4 import BeautifulSoup
from urllib.request import urlopen, Request
import pandas as pd
import re
import time

# 環境設定

In [89]:
NISHIOKA_URL = "https://www.atptour.com/en/players/yoshihito-nishioka/n732/player-stats?year=yyyy&surfaceType=all"
# headersとやらを変えないとはじかれるので
HEADERS = {
    "User-Agent": "camouflage useragent",
}

# 関数定義

# 分析

## read_htmlを使ってみる
- 西岡良仁の年ごとのサーブスタッツ表を読み込む

In [90]:
serve_data = []
for y in range(2014, 2024):
    url = NISHIOKA_URL.replace("yyyy", str(y))
    r = requests.get(url, headers=HEADERS)
    dfs = pd.read_html(r.text)
    serve_data.append(dfs[1])
    time.sleep(10)

In [91]:
serve_data[4]

,Singles Service Record,Singles Service Record.1
0,Aces,20
1,Double Faults,39
2,1st Serve,67%
3,1st Serve Points Won,61%
4,2nd Serve Points Won,50%
5,Break Points Faced,176
6,Break Points Saved,55%
7,Service Games Played,247
8,Service Games Won,68%
9,Total Service Points Won,58%


In [92]:
serve_stats = pd.DataFrame()
for year, df in zip(range(2014, 2024), serve_data):
    # 年ごとにレコードになるよう処理
    df = df.T.reset_index(drop=True).copy()
    df.columns = df.iloc[0]
    df = df.drop(df.index[0])
    # yearカラムを追加
    cols = df.columns
    df["year"] = year
    df = df[["year"] + cols.tolist()]
    # concat
    serve_stats = pd.concat([serve_stats, df])

In [93]:
serve_stats

,year,Aces,Double Faults,1st Serve,1st Serve Points Won,2nd Serve Points Won,Break Points Faced,Break Points Saved,Service Games Played,Service Games Won,Total Service Points Won
1,2014,1,5,55%,58%,33%,10,50%,9,44%,47%
1,2015,14,18,59%,60%,54%,66,52%,98,67%,57%
1,2016,43,45,60%,67%,54%,148,60%,229,74%,62%
1,2017,21,32,64%,64%,54%,131,65%,180,74%,61%
1,2018,20,39,67%,61%,50%,176,55%,247,68%,58%
1,2019,79,50,67%,63%,54%,326,61%,457,72%,60%
1,2020,56,36,67%,65%,56%,201,62%,324,77%,62%
1,2021,50,59,66%,63%,49%,246,52%,367,68%,58%
1,2022,81,38,67%,64%,55%,265,61%,430,76%,61%
1,2023,62,66,63%,64%,55%,215,52%,385,73%,61%


## BeautifulSoupを使ってみる
- 西岡良仁のATP公式サイトの1ページを取ってくる
- titleとbodyに分てみる

In [22]:
nishioka_url = "https://www.atptour.com/en/players/yoshihito-nishioka/n732/player-stats?year=2023&surfaceType=all"
# headersとやらを変えないとはじかれるので
headers = {
    "User-Agent": "camouflage useragent",
}
request = Request(nishioka_url, headers=headers)
html = urlopen(request) 
data = html.read()
html = data.decode('utf-8')

# HTMLを解析
soup = BeautifulSoup(html, 'html.parser')

# 解析したHTMLから任意の部分のみを抽出（ここではtitleとbody）
title = soup.find("title")
body = soup.find("body")

print("title: " + title.text)
print("body: " + body.text)

title: 
        Yoshihito Nishioka | Player Stats | ATP Tour | Tennis
    
body: 

















 
                        OneVision
                        


 
                        ATP Serves
                        


 
                        Watch
                        


 
                        Listen
                        



                        Newsletters
                    






EN
                        




                                    English
                                



                                    Spanish
                                


















                Menu
            




                    ATP Tour
                









            Scores
        

ATP Tour


Challenger


Results Archive


ATP WTA Live

 

            Stats
        

Landing


Leaderboards


Serve Tracker


Performance Zone


Win/Loss


Stats


#1s

 

            Rankings
        

Landing


Singles


Doubles


Singles Race


Doubles Race